# 01 - Reservoir Selection

Runs the staged reservoir search and writes the locked configuration used in the dissertation. See Methodology Section 3.1.3, Results Section 4.1.1, and Appendix D.

## 1. Imports and configuration

Sets the fixed seeds, repository paths and shared acoustic front end.

In [1]:

from pathlib import Path
import csv
import gc
import hashlib
import itertools
import json
import os
import random
import time

import joblib
import librosa
import numpy as np
import reservoirpy as rpy

from reservoirpy.nodes import Reservoir
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rpy.set_seed(SEED)
os.environ["RESERVOIRPY_VERBOSITY"] = "0"

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
def find_project_root() -> Path:
    """Find the repository root from the data/nsynth folder."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "nsynth").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/nsynth. Start Jupyter inside the repository."
    )


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "nsynth"
TRAIN_DIR = DATA_ROOT / "nsynth-train"
VALID_DIR = DATA_ROOT / "nsynth-valid"

CACHE_ROOT = PROJECT_ROOT / "cache_final"
RESULTS_DIR = PROJECT_ROOT / "results_sweep_final"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Shared acoustic front end
# Deltas are calculated after the static spectrogram has been fixed
# to MAX_T. The main notebook uses the identical procedure.
# ---------------------------------------------------------------------
SR = 16_000
N_MELS = 64
N_FFT = 1_024
HOP_LENGTH = 512
MAX_T = 125
USE_DELTAS = True
FEAT_DIM = N_MELS * (3 if USE_DELTAS else 1)
SCALER_N_EXAMPLES = 3_000

FEATURE_CFG = {
    "sr": SR,
    "n_mels": N_MELS,
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "max_t": MAX_T,
    "use_deltas": USE_DELTAS,
    "delta_after_length_fix": True,
}

# ---------------------------------------------------------------------
# Class scheme
# ---------------------------------------------------------------------
FAMILY_NAMES = [
    "bass", "brass", "flute", "guitar", "keyboard",
    "mallet", "organ", "reed", "string", "vocal",
]
N_CLASSES = len(FAMILY_NAMES)
SYNTH_LEAD_IDX = 9
FAMILY_MAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4,
              5: 5, 6: 6, 7: 7, 8: 8, 10: 9}

# ---------------------------------------------------------------------
# Cache controls
# ---------------------------------------------------------------------
FORCE_REBUILD_FEATURE_CACHE = False
FEATURE_CACHE_DTYPE = np.float32

# ---------------------------------------------------------------------
# Sweep design
# ---------------------------------------------------------------------
RC_CONNECTIVITY = 0.10
STAGE1_UNITS = 500

N_TRAIN_SUBSET = 4_000
N_VALID_SUBSET = 2_000

SR_GRID = [0.70, 0.80, 0.90, 0.95, 0.99]
LR_GRID = [0.05, 0.10, 0.15, 0.30]
BASE_IS_GRID = [0.01, 0.02, 0.05, 0.10]

# The original search selected the upper boundary 0.10. Rather than
# repeating every SR/LR pair at larger scales, the boundary is extended
# for the leading SR/LR combinations from the coarse search.
EXTENDED_IS_GRID = [0.20, 0.30, 0.50]
N_BOUNDARY_DYNAMICS = 5

TOP_K_DYNAMICS = 5
SIZE_GRID = [200, 500, 700, 1_000]

# Larger confirmation stage for the strongest complete configurations.
RUN_CONFIRMATION_STAGE = True
N_CONFIRM_CANDIDATES = 3
N_CONFIRM_TRAIN = 12_000
N_CONFIRM_VALID = None  # None uses the complete official validation split.

# ---------------------------------------------------------------------
# Ridge proxy readout
# This exact grid and solver setup are reused by the main notebook.
# ---------------------------------------------------------------------
RIDGE_ALPHAS = [
    1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5,
    1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0,
]
RIDGE_CLASS_WEIGHT = "balanced"
RIDGE_SOLVER = "lsqr"
RIDGE_TOL = 1e-6
RIDGE_MAX_ITER = 20_000
SELECTION_METRIC = "macro_f1"
TIE_ATOL = 1e-12


def stable_hash(obj) -> str:
    txt = json.dumps(obj, sort_keys=True, default=str)
    return hashlib.sha1(txt.encode("utf-8")).hexdigest()[:10]


FEATURE_TAG = stable_hash(FEATURE_CFG)
SCALER_CFG = {
    "feature_tag": FEATURE_TAG,
    "scaler_n_examples": SCALER_N_EXAMPLES,
    "seed": SEED,
}
SCALER_TAG = stable_hash(SCALER_CFG)

FEATURE_CACHE_DIR = CACHE_ROOT / f"mel_{FEATURE_TAG}"
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_SCALER_PATH = (
    FEATURE_CACHE_DIR / f"feature_scaler_{SCALER_TAG}.joblib"
)

print(f"Feature tag:         {FEATURE_TAG}")
print(f"Scaler tag:          {SCALER_TAG}")
print(f"Feature cache:       {FEATURE_CACHE_DIR}")
print(f"Sweep results:       {RESULTS_DIR}")
print(f"Coarse configurations: "
      f"{len(SR_GRID) * len(LR_GRID) * len(BASE_IS_GRID)}")


Feature tag:         8cfbd6d555
Scaler tag:          2e0a669124
Feature cache:       /home/olliechandler/ESN-NSYNTH/cache_final/mel_8cfbd6d555
Sweep results:       /home/olliechandler/ESN-NSYNTH/results_sweep_final
Coarse configurations: 80


## 2. Metadata and acoustic caches

Builds the training and validation feature caches used for reservoir selection. See Methodology Sections 3.1.1 and 3.1.2.

In [2]:

def load_metadata(split_dir: Path) -> dict:
    path = split_dir / "examples.json"
    if not path.exists():
        raise FileNotFoundError(f"Missing metadata: {path}")
    with open(path) as f:
        return json.load(f)


def keep_key(key: str, metadata: dict) -> bool:
    return metadata[key]["instrument_family"] != SYNTH_LEAD_IDX


def label_of(key: str, metadata: dict) -> int:
    return FAMILY_MAP[metadata[key]["instrument_family"]]


def split_keys(metadata: dict) -> list[str]:
    return sorted(k for k in metadata if keep_key(k, metadata))


def ordered_hash(values) -> str:
    payload = json.dumps(list(values), separators=(",", ":"))
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


train_meta = load_metadata(TRAIN_DIR)
valid_meta = load_metadata(VALID_DIR)
train_keys = split_keys(train_meta)
valid_keys = split_keys(valid_meta)

print(f"Training examples:   {len(train_keys):,}")
print(f"Validation examples: {len(valid_keys):,}")


Training examples:   283,704
Validation examples: 12,678


In [3]:

def audio_path(split_dir: Path, key: str) -> Path:
    return split_dir / "audio" / f"{key}.wav"


def load_audio(split_dir: Path, key: str) -> np.ndarray:
    path = audio_path(split_dir, key)
    if not path.exists():
        raise FileNotFoundError(f"Missing audio file: {path}")
    audio, _ = librosa.load(path, sr=SR, mono=True)
    return audio


def fix_static_length(static: np.ndarray, max_t: int = MAX_T) -> np.ndarray:
    # static shape: (time, mel)
    if static.shape[0] >= max_t:
        return static[:max_t]
    pad = np.zeros(
        (max_t - static.shape[0], static.shape[1]),
        dtype=static.dtype,
    )
    return np.vstack([static, pad])


def extract_feature(split_dir: Path, key: str) -> np.ndarray:
    audio = load_audio(split_dir, key)
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0,
    )
    static = librosa.power_to_db(mel, ref=np.max).T.astype(np.float32)
    static = fix_static_length(static)

    if not USE_DELTAS:
        return static

    d1 = librosa.feature.delta(static.T, order=1).T
    d2 = librosa.feature.delta(static.T, order=2).T
    return np.concatenate([static, d1, d2], axis=1).astype(np.float32)


def cache_paths(split_name: str):
    return (
        FEATURE_CACHE_DIR / f"X_{split_name}.npy",
        FEATURE_CACHE_DIR / f"y_{split_name}.npy",
        FEATURE_CACHE_DIR / f"keys_{split_name}.json",
        FEATURE_CACHE_DIR / f"manifest_{split_name}.json",
    )


def npy_cache_valid(path: Path, expected_shape: tuple, expected_dtype) -> bool:
    if not path.exists():
        return False
    try:
        arr = np.load(path, mmap_mode="r")
        valid = (
            arr.shape == expected_shape
            and arr.dtype == np.dtype(expected_dtype)
        )
        del arr
        return valid
    except Exception:
        return False


def cache_feature_split(
    split_name: str,
    split_dir: Path,
    keys: list[str],
    metadata: dict,
):
    X_path, y_path, keys_path, manifest_path = cache_paths(split_name)
    expected_key_hash = ordered_hash(keys)
    expected_label_hash = ordered_hash(
        [label_of(k, metadata) for k in keys]
    )

    valid = (
        npy_cache_valid(
            X_path,
            (len(keys), MAX_T, FEAT_DIM),
            FEATURE_CACHE_DTYPE,
        )
        and npy_cache_valid(y_path, (len(keys),), np.int64)
        and keys_path.exists()
        and manifest_path.exists()
    )

    if valid and not FORCE_REBUILD_FEATURE_CACHE:
        with open(keys_path) as f:
            cached_keys = json.load(f)
        with open(manifest_path) as f:
            manifest = json.load(f)

        valid = (
            cached_keys == keys
            and manifest.get("feature_cfg") == FEATURE_CFG
            and manifest.get("ordered_key_hash") == expected_key_hash
            and manifest.get("ordered_label_hash") == expected_label_hash
        )

    if valid and not FORCE_REBUILD_FEATURE_CACHE:
        print(f"Using feature cache for {split_name}: {X_path}")
        return X_path, y_path, keys_path

    print(f"Building feature cache for {split_name}...")
    X = np.lib.format.open_memmap(
        X_path,
        mode="w+",
        dtype=FEATURE_CACHE_DTYPE,
        shape=(len(keys), MAX_T, FEAT_DIM),
    )
    y = np.lib.format.open_memmap(
        y_path,
        mode="w+",
        dtype=np.int64,
        shape=(len(keys),),
    )

    started = time.time()
    for i, key in enumerate(keys):
        X[i] = extract_feature(split_dir, key)
        y[i] = label_of(key, metadata)

        if (i + 1) % 5_000 == 0 or i + 1 == len(keys):
            elapsed = time.time() - started
            rate = (i + 1) / max(elapsed, 1e-9)
            print(
                f"  {split_name:5s} {i+1:,}/{len(keys):,} "
                f"| {rate:.1f} examples/s"
            )
            X.flush()
            y.flush()

    with open(keys_path, "w") as f:
        json.dump(keys, f)
    with open(manifest_path, "w") as f:
        json.dump(
            {
                "split": split_name,
                "feature_cfg": FEATURE_CFG,
                "feature_tag": FEATURE_TAG,
                "ordered_key_hash": expected_key_hash,
                "ordered_label_hash": expected_label_hash,
                "n_examples": len(keys),
            },
            f,
            indent=2,
        )

    X.flush()
    y.flush()
    del X, y
    gc.collect()
    return X_path, y_path, keys_path


train_X_path, train_y_path, train_keys_path = cache_feature_split(
    "train", TRAIN_DIR, train_keys, train_meta
)
valid_X_path, valid_y_path, valid_keys_path = cache_feature_split(
    "valid", VALID_DIR, valid_keys, valid_meta
)


Building feature cache for train...
  train 5,000/283,704 | 460.3 examples/s
  train 10,000/283,704 | 457.6 examples/s
  train 15,000/283,704 | 464.2 examples/s
  train 20,000/283,704 | 466.2 examples/s
  train 25,000/283,704 | 470.4 examples/s
  train 30,000/283,704 | 475.5 examples/s
  train 35,000/283,704 | 478.2 examples/s
  train 40,000/283,704 | 480.2 examples/s
  train 45,000/283,704 | 481.7 examples/s
  train 50,000/283,704 | 484.1 examples/s
  train 55,000/283,704 | 485.3 examples/s
  train 60,000/283,704 | 485.9 examples/s
  train 65,000/283,704 | 486.3 examples/s
  train 70,000/283,704 | 487.1 examples/s
  train 75,000/283,704 | 487.4 examples/s
  train 80,000/283,704 | 486.6 examples/s
  train 85,000/283,704 | 486.5 examples/s
  train 90,000/283,704 | 487.2 examples/s
  train 95,000/283,704 | 487.5 examples/s
  train 100,000/283,704 | 487.4 examples/s
  train 105,000/283,704 | 488.0 examples/s
  train 110,000/283,704 | 488.1 examples/s
  train 115,000/283,704 | 488.6 exampl

## 3. Feature scaler and fixed subsets

Fits the training-only feature scaler and creates the fixed stratified subsets used by the search. See Methodology Section 3.1.2 and Appendix C.

In [4]:

def fit_or_load_feature_scaler() -> StandardScaler:
    if FEATURE_SCALER_PATH.exists() and not FORCE_REBUILD_FEATURE_CACHE:
        print(f"Loading feature scaler: {FEATURE_SCALER_PATH}")
        return joblib.load(FEATURE_SCALER_PATH)

    print("Fitting feature scaler from training data only...")
    X = np.load(train_X_path, mmap_mode="r")
    rng = np.random.default_rng(SEED)
    n = min(SCALER_N_EXAMPLES, len(X))
    indices = rng.choice(len(X), size=n, replace=False)

    scaler = StandardScaler()
    for start in range(0, len(indices), 64):
        batch = indices[start:start + 64]
        frames = np.asarray(
            X[batch], dtype=np.float32
        ).reshape(-1, FEAT_DIM)
        scaler.partial_fit(frames)

    joblib.dump(scaler, FEATURE_SCALER_PATH)
    del X
    gc.collect()
    print(f"Saved feature scaler: {FEATURE_SCALER_PATH}")
    return scaler


feature_scaler = fit_or_load_feature_scaler()

y_train_all = np.asarray(
    np.load(train_y_path, mmap_mode="r"), dtype=np.int64
)
y_valid_all = np.asarray(
    np.load(valid_y_path, mmap_mode="r"), dtype=np.int64
)


def stratified_indices(y: np.ndarray, n: int | None, seed: int) -> np.ndarray:
    if n is None or n >= len(y):
        return np.arange(len(y), dtype=np.int64)

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=n,
        random_state=seed,
    )
    indices, _ = next(splitter.split(np.zeros(len(y)), y))
    return np.sort(indices.astype(np.int64))


train_idx = stratified_indices(y_train_all, N_TRAIN_SUBSET, SEED)
valid_idx = stratified_indices(y_valid_all, N_VALID_SUBSET, SEED + 1)

confirm_train_idx = stratified_indices(
    y_train_all, N_CONFIRM_TRAIN, SEED + 2
)
confirm_valid_idx = stratified_indices(
    y_valid_all, N_CONFIRM_VALID, SEED + 3
)

np.save(RESULTS_DIR / "stage_train_indices.npy", train_idx)
np.save(RESULTS_DIR / "stage_valid_indices.npy", valid_idx)
np.save(RESULTS_DIR / "confirm_train_indices.npy", confirm_train_idx)
np.save(RESULTS_DIR / "confirm_valid_indices.npy", confirm_valid_idx)

print(f"Stage training subset:      {len(train_idx):,}")
print(f"Stage validation subset:    {len(valid_idx):,}")
print(f"Confirm training subset:    {len(confirm_train_idx):,}")
print(f"Confirm validation subset:  {len(confirm_valid_idx):,}")


Fitting feature scaler from training data only...
Saved feature scaler: /home/olliechandler/ESN-NSYNTH/cache_final/mel_8cfbd6d555/feature_scaler_2e0a669124.joblib
Stage training subset:      4,000
Stage validation subset:    2,000
Confirm training subset:    12,000
Confirm validation subset:  12,678


## 4. Reservoir and Ridge evaluation

Defines the reservoir, pooled representation and Ridge evaluation used throughout the search. See Methodology Section 3.1.3.

In [5]:

def make_reservoir(units: int, sr_: float, lr_: float, is_: float):
    reservoir = Reservoir(
        units=units,
        sr=sr_,
        lr=lr_,
        input_scaling=is_,
        rc_connectivity=RC_CONNECTIVITY,
        seed=SEED,
    )
    reservoir.run(np.zeros((3, FEAT_DIM), dtype=np.float32))
    return reservoir


def pooled_reservoir_states(
    X_path: Path,
    indices: np.ndarray,
    units: int,
    sr_: float,
    lr_: float,
    is_: float,
) -> np.ndarray:
    X = np.load(X_path, mmap_mode="r")
    reservoir = make_reservoir(units, sr_, lr_, is_)
    pooled = np.empty((len(indices), units * 3), dtype=np.float32)

    for j, i in enumerate(indices):
        features = np.asarray(X[i], dtype=np.float32)
        features = feature_scaler.transform(features)
        reservoir.reset()
        states = np.asarray(
            reservoir.run(features), dtype=np.float32
        )
        pooled[j] = np.concatenate(
            [states.mean(axis=0), states.max(axis=0), states[-1]],
            axis=0,
        )

    del X, reservoir
    gc.collect()
    return pooled


def metric_bundle(labels, predictions) -> dict:
    return {
        "acc": float(accuracy_score(labels, predictions)),
        "macro_f1": float(
            f1_score(labels, predictions, average="macro")
        ),
        "balanced_acc": float(
            balanced_accuracy_score(labels, predictions)
        ),
    }


def choose_alpha(alpha_rows: list[dict]) -> dict:
    best_score = max(r[SELECTION_METRIC] for r in alpha_rows)
    tied = [
        r for r in alpha_rows
        if np.isclose(
            r[SELECTION_METRIC],
            best_score,
            rtol=0.0,
            atol=TIE_ATOL,
        )
    ]
    # Prefer the more regularised model when validation predictions tie.
    return max(tied, key=lambda r: r["alpha"])


def evaluate_config(
    units: int,
    sr_: float,
    lr_: float,
    is_: float,
    tr_idx: np.ndarray,
    va_idx: np.ndarray,
    stage: str,
):
    Xtr = pooled_reservoir_states(
        train_X_path, tr_idx, units, sr_, lr_, is_
    )
    Xva = pooled_reservoir_states(
        valid_X_path, va_idx, units, sr_, lr_, is_
    )

    ytr = y_train_all[tr_idx]
    yva = y_valid_all[va_idx]

    summary_scaler = StandardScaler().fit(Xtr)
    Xtr_s = summary_scaler.transform(Xtr)
    Xva_s = summary_scaler.transform(Xva)

    alpha_rows = []
    for alpha in RIDGE_ALPHAS:
        classifier = RidgeClassifier(
            alpha=alpha,
            class_weight=RIDGE_CLASS_WEIGHT,
            solver=RIDGE_SOLVER,
            tol=RIDGE_TOL,
            max_iter=RIDGE_MAX_ITER,
        )
        classifier.fit(Xtr_s, ytr)
        prediction = classifier.predict(Xva_s)
        alpha_rows.append(
            {
                "stage": stage,
                "units": int(units),
                "sr": float(sr_),
                "lr": float(lr_),
                "input_scaling": float(is_),
                "alpha": float(alpha),
                **metric_bundle(yva, prediction),
            }
        )

    selected = choose_alpha(alpha_rows)
    result = {
        "stage": stage,
        "units": int(units),
        "sr": float(sr_),
        "lr": float(lr_),
        "input_scaling": float(is_),
        "alpha": float(selected["alpha"]),
        "val_acc": float(selected["acc"]),
        "val_macro_f1": float(selected["macro_f1"]),
        "val_balanced_acc": float(selected["balanced_acc"]),
    }

    del Xtr, Xva, Xtr_s, Xva_s, summary_scaler
    gc.collect()
    return result, alpha_rows


def write_rows(path: Path, rows: list[dict]):
    if not rows:
        raise ValueError(f"No rows available for {path}")
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved: {path}")


## 5. Stage 1A - coarse dynamics search

Runs the coarse spectral-radius, leak-rate and input-scaling search reported in Appendix D.

In [6]:

stage1_rows = []
stage1_alpha_rows = []
started = time.time()

coarse_configs = list(
    itertools.product(SR_GRID, LR_GRID, BASE_IS_GRID)
)

for n, (sr_, lr_, is_) in enumerate(coarse_configs, start=1):
    result, alpha_rows = evaluate_config(
        units=STAGE1_UNITS,
        sr_=sr_,
        lr_=lr_,
        is_=is_,
        tr_idx=train_idx,
        va_idx=valid_idx,
        stage="stage1_coarse",
    )
    stage1_rows.append(result)
    stage1_alpha_rows.extend(alpha_rows)

    best_so_far = max(
        stage1_rows, key=lambda r: r["val_macro_f1"]
    )
    marker = " <- best" if result is best_so_far else ""
    print(
        f"[{n:>3}/{len(coarse_configs)}] "
        f"sr={sr_:.2f} lr={lr_:.2f} is={is_:.2f} "
        f"| macro F1={result['val_macro_f1']*100:.2f}% "
        f"| alpha={result['alpha']:.1e} "
        f"| elapsed={time.time()-started:.0f}s"
        f"{marker}"
    )

write_rows(
    RESULTS_DIR / "reservoir_stage1_coarse.csv",
    stage1_rows,
)
write_rows(
    RESULTS_DIR / "reservoir_stage1_coarse_alpha_curves.csv",
    stage1_alpha_rows,
)


[  1/80] sr=0.70 lr=0.05 is=0.01 | macro F1=49.45% | alpha=1.0e+02 | elapsed=271s <- best
[  2/80] sr=0.70 lr=0.05 is=0.02 | macro F1=49.22% | alpha=1.0e+02 | elapsed=484s
[  3/80] sr=0.70 lr=0.05 is=0.05 | macro F1=49.81% | alpha=1.0e+02 | elapsed=582s <- best
[  4/80] sr=0.70 lr=0.05 is=0.10 | macro F1=50.50% | alpha=1.0e+02 | elapsed=643s <- best
[  5/80] sr=0.70 lr=0.10 is=0.01 | macro F1=49.77% | alpha=1.0e-03 | elapsed=985s
[  6/80] sr=0.70 lr=0.10 is=0.02 | macro F1=49.84% | alpha=1.0e+02 | elapsed=1208s
[  7/80] sr=0.70 lr=0.10 is=0.05 | macro F1=50.01% | alpha=1.0e+02 | elapsed=1327s
[  8/80] sr=0.70 lr=0.10 is=0.10 | macro F1=51.79% | alpha=1.0e+01 | elapsed=1396s <- best
[  9/80] sr=0.70 lr=0.15 is=0.01 | macro F1=48.51% | alpha=1.0e+02 | elapsed=1683s
[ 10/80] sr=0.70 lr=0.15 is=0.02 | macro F1=48.95% | alpha=1.0e+01 | elapsed=1924s
[ 11/80] sr=0.70 lr=0.15 is=0.05 | macro F1=50.36% | alpha=1.0e+01 | elapsed=2046s
[ 12/80] sr=0.70 lr=0.15 is=0.10 | macro F1=50.83% | alpha=1

## 6. Stage 1B - boundary extension

Extends input scaling only where the Stage 1A result reached the tested boundary. See Appendix D.

In [7]:

# Identify the leading distinct (spectral radius, leak rate) pairs.
pair_best = {}
for row in stage1_rows:
    pair = (row["sr"], row["lr"])
    if (
        pair not in pair_best
        or row["val_macro_f1"] > pair_best[pair]["val_macro_f1"]
    ):
        pair_best[pair] = row

leading_pairs = [
    (r["sr"], r["lr"])
    for r in sorted(
        pair_best.values(),
        key=lambda r: r["val_macro_f1"],
        reverse=True,
    )[:N_BOUNDARY_DYNAMICS]
]

boundary_rows = []
boundary_alpha_rows = []
boundary_configs = [
    (sr_, lr_, is_)
    for sr_, lr_ in leading_pairs
    for is_ in EXTENDED_IS_GRID
]

for n, (sr_, lr_, is_) in enumerate(boundary_configs, start=1):
    result, alpha_rows = evaluate_config(
        units=STAGE1_UNITS,
        sr_=sr_,
        lr_=lr_,
        is_=is_,
        tr_idx=train_idx,
        va_idx=valid_idx,
        stage="stage1_boundary",
    )
    boundary_rows.append(result)
    boundary_alpha_rows.extend(alpha_rows)
    print(
        f"[{n:>2}/{len(boundary_configs)}] "
        f"sr={sr_:.2f} lr={lr_:.2f} is={is_:.2f} "
        f"| macro F1={result['val_macro_f1']*100:.2f}% "
        f"| alpha={result['alpha']:.1e}"
    )

write_rows(
    RESULTS_DIR / "reservoir_stage1_boundary.csv",
    boundary_rows,
)
write_rows(
    RESULTS_DIR / "reservoir_stage1_boundary_alpha_curves.csv",
    boundary_alpha_rows,
)

all_dynamics_rows = stage1_rows + boundary_rows
top_dynamics = sorted(
    all_dynamics_rows,
    key=lambda r: r["val_macro_f1"],
    reverse=True,
)[:TOP_K_DYNAMICS]

print("\nLeading dynamics configurations")
for rank, row in enumerate(top_dynamics, start=1):
    print(
        f"{rank}. sr={row['sr']:.2f} lr={row['lr']:.2f} "
        f"is={row['input_scaling']:.2f} "
        f"macro F1={row['val_macro_f1']*100:.2f}%"
    )


[ 1/15] sr=0.80 lr=0.10 is=0.20 | macro F1=51.70% | alpha=1.0e+02
[ 2/15] sr=0.80 lr=0.10 is=0.30 | macro F1=51.94% | alpha=1.0e+02
[ 3/15] sr=0.80 lr=0.10 is=0.50 | macro F1=51.35% | alpha=1.0e+02
[ 4/15] sr=0.99 lr=0.30 is=0.20 | macro F1=50.63% | alpha=1.0e+02
[ 5/15] sr=0.99 lr=0.30 is=0.30 | macro F1=48.95% | alpha=1.0e+02
[ 6/15] sr=0.99 lr=0.30 is=0.50 | macro F1=46.63% | alpha=1.0e+02
[ 7/15] sr=0.70 lr=0.10 is=0.20 | macro F1=51.23% | alpha=1.0e+02
[ 8/15] sr=0.70 lr=0.10 is=0.30 | macro F1=51.45% | alpha=1.0e+02
[ 9/15] sr=0.70 lr=0.10 is=0.50 | macro F1=50.63% | alpha=1.0e+02
[10/15] sr=0.95 lr=0.10 is=0.20 | macro F1=51.56% | alpha=1.0e+02
[11/15] sr=0.95 lr=0.10 is=0.30 | macro F1=51.62% | alpha=1.0e+02
[12/15] sr=0.95 lr=0.10 is=0.50 | macro F1=51.45% | alpha=1.0e+02
[13/15] sr=0.99 lr=0.05 is=0.20 | macro F1=52.59% | alpha=1.0e+01
[14/15] sr=0.99 lr=0.05 is=0.30 | macro F1=52.59% | alpha=1.0e+02
[15/15] sr=0.99 lr=0.05 is=0.50 | macro F1=52.08% | alpha=1.0e+02
Saved: /ho

## 7. Stage 2 - reservoir size

Tests the leading dynamics settings across the reservoir sizes reported in Appendix D.

In [8]:

stage2_rows = []
stage2_alpha_rows = []
stage2_configs = [
    (
        int(units),
        float(dynamics["sr"]),
        float(dynamics["lr"]),
        float(dynamics["input_scaling"]),
    )
    for dynamics in top_dynamics
    for units in SIZE_GRID
]

for n, (units, sr_, lr_, is_) in enumerate(stage2_configs, start=1):
    result, alpha_rows = evaluate_config(
        units=units,
        sr_=sr_,
        lr_=lr_,
        is_=is_,
        tr_idx=train_idx,
        va_idx=valid_idx,
        stage="stage2_size",
    )
    stage2_rows.append(result)
    stage2_alpha_rows.extend(alpha_rows)
    print(
        f"[{n:>2}/{len(stage2_configs)}] "
        f"units={units:>4} sr={sr_:.2f} lr={lr_:.2f} "
        f"is={is_:.2f} "
        f"| macro F1={result['val_macro_f1']*100:.2f}%"
    )

write_rows(
    RESULTS_DIR / "reservoir_stage2_size.csv",
    stage2_rows,
)
write_rows(
    RESULTS_DIR / "reservoir_stage2_size_alpha_curves.csv",
    stage2_alpha_rows,
)

top_complete = sorted(
    stage2_rows,
    key=lambda r: r["val_macro_f1"],
    reverse=True,
)[:N_CONFIRM_CANDIDATES]

print("\nLeading complete reservoir configurations")
for rank, row in enumerate(top_complete, start=1):
    print(
        f"{rank}. units={row['units']} sr={row['sr']:.2f} "
        f"lr={row['lr']:.2f} is={row['input_scaling']:.2f} "
        f"macro F1={row['val_macro_f1']*100:.2f}%"
    )


[ 1/20] units= 200 sr=0.99 lr=0.05 is=0.20 | macro F1=51.52%
[ 2/20] units= 500 sr=0.99 lr=0.05 is=0.20 | macro F1=52.59%
[ 3/20] units= 700 sr=0.99 lr=0.05 is=0.20 | macro F1=55.61%
[ 4/20] units=1000 sr=0.99 lr=0.05 is=0.20 | macro F1=56.14%
[ 5/20] units= 200 sr=0.99 lr=0.05 is=0.30 | macro F1=50.36%
[ 6/20] units= 500 sr=0.99 lr=0.05 is=0.30 | macro F1=52.59%
[ 7/20] units= 700 sr=0.99 lr=0.05 is=0.30 | macro F1=55.65%
[ 8/20] units=1000 sr=0.99 lr=0.05 is=0.30 | macro F1=55.55%
[ 9/20] units= 200 sr=0.80 lr=0.10 is=0.10 | macro F1=48.86%
[10/20] units= 500 sr=0.80 lr=0.10 is=0.10 | macro F1=52.23%
[11/20] units= 700 sr=0.80 lr=0.10 is=0.10 | macro F1=56.01%
[12/20] units=1000 sr=0.80 lr=0.10 is=0.10 | macro F1=54.65%
[13/20] units= 200 sr=0.99 lr=0.05 is=0.50 | macro F1=49.74%
[14/20] units= 500 sr=0.99 lr=0.05 is=0.50 | macro F1=52.08%
[15/20] units= 700 sr=0.99 lr=0.05 is=0.50 | macro F1=54.37%
[16/20] units=1000 sr=0.99 lr=0.05 is=0.50 | macro F1=52.62%
[17/20] units= 200 sr=0.

## 8. Stage 3A - confirmation

Re-evaluates the final candidate configurations on the larger confirmation subsets. See Appendix D.3.

In [9]:

confirmation_rows = []
confirmation_alpha_rows = []

if RUN_CONFIRMATION_STAGE:
    for n, candidate in enumerate(top_complete, start=1):
        result, alpha_rows = evaluate_config(
            units=int(candidate["units"]),
            sr_=float(candidate["sr"]),
            lr_=float(candidate["lr"]),
            is_=float(candidate["input_scaling"]),
            tr_idx=confirm_train_idx,
            va_idx=confirm_valid_idx,
            stage="stage3_confirmation",
        )
        confirmation_rows.append(result)
        confirmation_alpha_rows.extend(alpha_rows)
        print(
            f"[{n}/{len(top_complete)}] "
            f"units={result['units']} sr={result['sr']:.2f} "
            f"lr={result['lr']:.2f} "
            f"is={result['input_scaling']:.2f} "
            f"| confirmed macro F1="
            f"{result['val_macro_f1']*100:.2f}%"
        )

    write_rows(
        RESULTS_DIR / "reservoir_stage3_confirmation.csv",
        confirmation_rows,
    )
    write_rows(
        RESULTS_DIR / "reservoir_stage3_confirmation_alpha_curves.csv",
        confirmation_alpha_rows,
    )
    selected_row = max(
        confirmation_rows,
        key=lambda r: r["val_macro_f1"],
    )
else:
    selected_row = max(
        stage2_rows,
        key=lambda r: r["val_macro_f1"],
    )

print("\nSelected reservoir")
print(json.dumps(selected_row, indent=2))


[1/3] units=1000 sr=0.99 lr=0.05 is=0.20 | confirmed macro F1=60.17%
[2/3] units=700 sr=0.80 lr=0.10 is=0.10 | confirmed macro F1=57.53%
[3/3] units=700 sr=0.99 lr=0.05 is=0.30 | confirmed macro F1=58.47%
Saved: /home/olliechandler/ESN-NSYNTH/results_sweep_final/reservoir_stage3_confirmation.csv
Saved: /home/olliechandler/ESN-NSYNTH/results_sweep_final/reservoir_stage3_confirmation_alpha_curves.csv

Selected reservoir
{
  "stage": "stage3_confirmation",
  "units": 1000,
  "sr": 0.99,
  "lr": 0.05,
  "input_scaling": 0.2,
  "alpha": 100.0,
  "val_acc": 0.6265183782931062,
  "val_macro_f1": 0.6017123878077728,
  "val_balanced_acc": 0.6324534914584826
}


## 9. Lock the selected configuration

Writes the locked reservoir manifest consumed by Notebook 02. See Results Section 4.1.1 and Appendix D.1.

In [10]:

SELECTED_RESERVOIR_CFG = {
    "units": int(selected_row["units"]),
    "sr": float(selected_row["sr"]),
    "lr": float(selected_row["lr"]),
    "input_scaling": float(selected_row["input_scaling"]),
    "rc_connectivity": float(RC_CONNECTIVITY),
}

selection_manifest = {
    "schema_version": 1,
    "selection_status": "locked",
    "selection_metric": SELECTION_METRIC,
    "reservoir_seed": SEED,
    "feature_cfg": FEATURE_CFG,
    "feature_tag": FEATURE_TAG,
    "scaler_cfg": SCALER_CFG,
    "scaler_tag": SCALER_TAG,
    "feature_scaler_path": str(FEATURE_SCALER_PATH.relative_to(PROJECT_ROOT)),
    "reservoir_cfg": SELECTED_RESERVOIR_CFG,
    "selected_validation": {
        "acc": float(selected_row["val_acc"]),
        "macro_f1": float(selected_row["val_macro_f1"]),
        "balanced_acc": float(selected_row["val_balanced_acc"]),
        "ridge_alpha": float(selected_row["alpha"]),
        "stage": selected_row["stage"],
    },
    "search_design": {
        "stage1_units": STAGE1_UNITS,
        "train_subset_n": len(train_idx),
        "validation_subset_n": len(valid_idx),
        "sr_grid": SR_GRID,
        "lr_grid": LR_GRID,
        "base_input_scaling_grid": BASE_IS_GRID,
        "extended_input_scaling_grid": EXTENDED_IS_GRID,
        "boundary_dynamics_pairs": leading_pairs,
        "top_k_dynamics": TOP_K_DYNAMICS,
        "size_grid": SIZE_GRID,
        "confirmation_enabled": RUN_CONFIRMATION_STAGE,
        "confirmation_train_n": len(confirm_train_idx),
        "confirmation_valid_n": len(confirm_valid_idx),
        "ridge_alphas": RIDGE_ALPHAS,
        "ridge_solver": RIDGE_SOLVER,
        "ridge_tol": RIDGE_TOL,
        "ridge_max_iter": RIDGE_MAX_ITER,
    },
}

manifest_path = RESULTS_DIR / "selected_reservoir_config.json"
with open(manifest_path, "w") as f:
    json.dump(selection_manifest, f, indent=2)

print(f"Locked manifest: {manifest_path}")
print(json.dumps(SELECTED_RESERVOIR_CFG, indent=2))


Locked manifest: /home/olliechandler/ESN-NSYNTH/results_sweep_final/selected_reservoir_config.json
{
  "units": 1000,
  "sr": 0.99,
  "lr": 0.05,
  "input_scaling": 0.2,
  "rc_connectivity": 0.1
}


## 10. Stage 1 to 3A checks

Checks that the original search stages and locked manifest were written successfully.

In [11]:

checks = {
    "training_feature_cache_exists": Path(train_X_path).exists(),
    "validation_feature_cache_exists": Path(valid_X_path).exists(),
    "feature_scaler_exists": FEATURE_SCALER_PATH.exists(),
    "coarse_results_exist": (
        RESULTS_DIR / "reservoir_stage1_coarse.csv"
    ).exists(),
    "boundary_results_exist": (
        RESULTS_DIR / "reservoir_stage1_boundary.csv"
    ).exists(),
    "size_results_exist": (
        RESULTS_DIR / "reservoir_stage2_size.csv"
    ).exists(),
    "selection_manifest_exists": manifest_path.exists(),
    "selected_input_scaling_not_unchecked_boundary": (
        selected_row["input_scaling"] < max(EXTENDED_IS_GRID)
    ),
}

if RUN_CONFIRMATION_STAGE:
    checks["confirmation_results_exist"] = (
        RESULTS_DIR / "reservoir_stage3_confirmation.csv"
    ).exists()

print("Final completion checks")
for name, passed in checks.items():
    print(f"  {name:48s}: {passed}")

if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise RuntimeError(
        "Reservoir-selection notebook is not complete. "
        f"Failed checks: {failed}"
    )

Final completion checks
  training_feature_cache_exists                   : True
  validation_feature_cache_exists                 : True
  feature_scaler_exists                           : True
  coarse_results_exist                            : True
  boundary_results_exist                          : True
  size_results_exist                              : True
  selection_manifest_exists                       : True
  selected_input_scaling_not_unchecked_boundary   : True
  confirmation_results_exist                      : True


## 11. Stage 3B - Ridge alpha extension

Extends the Ridge grid only for the three confirmed candidates because Stage 3A selected the original upper boundary. See Appendix D.4.

In [12]:
# =====================================================================
# TARGETED RIDGE RANGE EXTENSION
# =====================================================================

import csv
import gc
import json

import numpy as np

from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------------------
# Larger Ridge values to test
# ---------------------------------------------------------------------
EXTENSION_ALPHAS = [
    100.0,
    300.0,
    1_000.0,
    3_000.0,
    10_000.0,
]


# ---------------------------------------------------------------------
# The three candidates selected before the Ridge-range extension
# ---------------------------------------------------------------------
CONFIRMED_CANDIDATES = [
    {
        "units": 1_000,
        "sr": 0.99,
        "lr": 0.05,
        "input_scaling": 0.20,
    },
    {
        "units": 700,
        "sr": 0.80,
        "lr": 0.10,
        "input_scaling": 0.10,
    },
    {
        "units": 700,
        "sr": 0.99,
        "lr": 0.05,
        "input_scaling": 0.30,
    },
]


# ---------------------------------------------------------------------
# Retain pooled summaries so further Ridge extensions remain cheap
# ---------------------------------------------------------------------
EXTENSION_CACHE_DIR = (
    RESULTS_DIR / "confirmation_pooled_summaries"
)
EXTENSION_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# Reload the exact confirmation indices used previously
# ---------------------------------------------------------------------
confirm_train_idx = np.load(
    RESULTS_DIR / "confirm_train_indices.npy"
)

confirm_valid_idx = np.load(
    RESULTS_DIR / "confirm_valid_indices.npy"
)


# ---------------------------------------------------------------------
# Load the original confirmation results
# ---------------------------------------------------------------------
original_confirmation_path = (
    RESULTS_DIR / "reservoir_stage3_confirmation.csv"
)

if not original_confirmation_path.exists():
    raise FileNotFoundError(
        "Original confirmation results were not found: "
        f"{original_confirmation_path}"
    )

with open(
    original_confirmation_path,
    newline="",
) as file:
    original_confirmation_rows = list(
        csv.DictReader(file)
    )


def candidate_key(row):
    """Create a stable key for matching candidate configurations."""
    return (
        int(float(row["units"])),
        round(float(row["sr"]), 8),
        round(float(row["lr"]), 8),
        round(float(row["input_scaling"]), 8),
    )


original_confirmation = {
    candidate_key(row): row
    for row in original_confirmation_rows
}


def select_extended_alpha(rows):
    """
    Select using the pre-specified validation metric.

    When multiple alphas produce numerically identical validation
    scores, prefer the more strongly regularised model.
    """
    best_score = max(
        row[SELECTION_METRIC]
        for row in rows
    )

    tied = [
        row
        for row in rows
        if np.isclose(
            row[SELECTION_METRIC],
            best_score,
            rtol=0.0,
            atol=TIE_ATOL,
        )
    ]

    return max(
        tied,
        key=lambda row: row["alpha"],
    )


extension_alpha_rows = []
extension_candidate_rows = []


# =====================================================================
# Evaluate each preselected confirmation candidate
# =====================================================================
for candidate_number, candidate in enumerate(
    CONFIRMED_CANDIDATES,
    start=1,
):
    units = int(candidate["units"])
    sr_ = float(candidate["sr"])
    lr_ = float(candidate["lr"])
    is_ = float(candidate["input_scaling"])

    key = candidate_key(candidate)

    if key not in original_confirmation:
        raise KeyError(
            "Candidate was not found in the original "
            "confirmation results:\n"
            f"{candidate}"
        )

    old_row = original_confirmation[key]
    old_alpha = float(old_row["alpha"])
    old_macro_f1 = float(
        old_row["val_macro_f1"]
    )

    # Repeat the candidate's actual original alpha, then test the
    # additional larger values. This avoids incorrectly assuming
    # that every original candidate selected alpha=100.
    candidate_alphas = sorted(
        set(
            [
                old_alpha,
                *EXTENSION_ALPHAS,
            ]
        )
    )

    config_tag = stable_hash(candidate)

    train_summary_path = (
        EXTENSION_CACHE_DIR
        / f"Xtr_pooled_{config_tag}.npy"
    )

    valid_summary_path = (
        EXTENSION_CACHE_DIR
        / f"Xva_pooled_{config_tag}.npy"
    )

    print()
    print("=" * 70)
    print(
        f"Candidate {candidate_number}/"
        f"{len(CONFIRMED_CANDIDATES)}"
    )
    print(
        f"units={units}, "
        f"sr={sr_}, "
        f"lr={lr_}, "
        f"input scaling={is_}"
    )
    print(
        f"Original selected alpha={old_alpha:.1e}, "
        f"macro F1={old_macro_f1 * 100:.4f}%"
    )
    print("=" * 70)

    # -----------------------------------------------------------------
    # Load or generate training summaries
    # -----------------------------------------------------------------
    if train_summary_path.exists():
        print(
            "Loading cached training summaries..."
        )

        Xtr = np.load(
            train_summary_path,
            mmap_mode="r",
        )

    else:
        print(
            "Generating training summaries..."
        )

        Xtr_generated = pooled_reservoir_states(
            train_X_path,
            confirm_train_idx,
            units,
            sr_,
            lr_,
            is_,
        )

        np.save(
            train_summary_path,
            Xtr_generated,
        )

        del Xtr_generated
        gc.collect()

        Xtr = np.load(
            train_summary_path,
            mmap_mode="r",
        )

    # -----------------------------------------------------------------
    # Load or generate validation summaries
    # -----------------------------------------------------------------
    if valid_summary_path.exists():
        print(
            "Loading cached validation summaries..."
        )

        Xva = np.load(
            valid_summary_path,
            mmap_mode="r",
        )

    else:
        print(
            "Generating validation summaries..."
        )

        Xva_generated = pooled_reservoir_states(
            valid_X_path,
            confirm_valid_idx,
            units,
            sr_,
            lr_,
            is_,
        )

        np.save(
            valid_summary_path,
            Xva_generated,
        )

        del Xva_generated
        gc.collect()

        Xva = np.load(
            valid_summary_path,
            mmap_mode="r",
        )

    ytr = y_train_all[confirm_train_idx]
    yva = y_valid_all[confirm_valid_idx]

    # -----------------------------------------------------------------
    # Fit the pooled-summary scaler using training data only
    # -----------------------------------------------------------------
    summary_scaler = StandardScaler()

    Xtr_scaled = summary_scaler.fit_transform(
        np.asarray(
            Xtr,
            dtype=np.float32,
        )
    )

    Xva_scaled = summary_scaler.transform(
        np.asarray(
            Xva,
            dtype=np.float32,
        )
    )

    candidate_alpha_rows = []

    # -----------------------------------------------------------------
    # Evaluate the original alpha and the extended values
    # -----------------------------------------------------------------
    for alpha in candidate_alphas:
        classifier = RidgeClassifier(
            alpha=alpha,
            class_weight=RIDGE_CLASS_WEIGHT,
            solver=RIDGE_SOLVER,
            tol=RIDGE_TOL,
            max_iter=RIDGE_MAX_ITER,
        )

        classifier.fit(
            Xtr_scaled,
            ytr,
        )

        predictions = classifier.predict(
            Xva_scaled
        )

        metrics = metric_bundle(
            yva,
            predictions,
        )

        row = {
            "stage": "stage3_ridge_extension",
            "units": units,
            "sr": sr_,
            "lr": lr_,
            "input_scaling": is_,
            "alpha": float(alpha),
            **metrics,
        }

        candidate_alpha_rows.append(row)
        extension_alpha_rows.append(row)

        print(
            f"alpha={alpha:>10.1e} | "
            f"accuracy={metrics['acc'] * 100:6.2f}% | "
            f"macro F1={metrics['macro_f1'] * 100:6.2f}% | "
            f"balanced={metrics['balanced_acc'] * 100:6.2f}%"
        )

    # -----------------------------------------------------------------
    # Reproduce the candidate's actual original selected alpha
    # -----------------------------------------------------------------
    repeated_original = next(
        row
        for row in candidate_alpha_rows
        if np.isclose(
            row["alpha"],
            old_alpha,
            rtol=0.0,
            atol=0.0,
        )
    )

    difference = abs(
        repeated_original["macro_f1"]
        - old_macro_f1
    )

    print(
        f"Original versus repeated alpha={old_alpha:.1e} "
        f"macro F1 difference: {difference:.12f}"
    )

    if difference > 1e-6:
        raise RuntimeError(
            "The repeated result at the candidate's original "
            "selected alpha does not match the original "
            "confirmation result. Stop before updating the "
            "selected configuration."
        )

    # -----------------------------------------------------------------
    # Select the best alpha using validation macro F1
    # -----------------------------------------------------------------
    selected = select_extended_alpha(
        candidate_alpha_rows
    )

    selected_candidate = {
        "stage": "stage3_ridge_extension",
        "units": units,
        "sr": sr_,
        "lr": lr_,
        "input_scaling": is_,
        "alpha": float(
            selected["alpha"]
        ),
        "val_acc": float(
            selected["acc"]
        ),
        "val_macro_f1": float(
            selected["macro_f1"]
        ),
        "val_balanced_acc": float(
            selected["balanced_acc"]
        ),
    }

    extension_candidate_rows.append(
        selected_candidate
    )

    print(
        "Selected for this candidate: "
        f"alpha={selected_candidate['alpha']:.1e}, "
        f"macro F1="
        f"{selected_candidate['val_macro_f1'] * 100:.2f}%"
    )

    del (
        Xtr,
        Xva,
        Xtr_scaled,
        Xva_scaled,
        summary_scaler,
        classifier,
        predictions,
    )

    gc.collect()


# =====================================================================
# Save complete extension results
# =====================================================================
write_rows(
    RESULTS_DIR
    / "reservoir_stage3_ridge_extension_alpha_curves.csv",
    extension_alpha_rows,
)

write_rows(
    RESULTS_DIR
    / "reservoir_stage3_ridge_extension.csv",
    extension_candidate_rows,
)


# ---------------------------------------------------------------------
# Select the final reservoir candidate
# ---------------------------------------------------------------------
extended_selected_row = max(
    extension_candidate_rows,
    key=lambda row: row["val_macro_f1"],
)

extension_summary = {
    "selection_metric": SELECTION_METRIC,
    "extension_alphas": EXTENSION_ALPHAS,
    "selected_candidate": extended_selected_row,
    "all_candidates": extension_candidate_rows,
    "pooled_summary_cache": str(
        EXTENSION_CACHE_DIR
    ),
}


recommendation_path = (
    RESULTS_DIR
    / "ridge_extension_recommendation.json"
)

with open(
    recommendation_path,
    "w",
) as file:
    json.dump(
        extension_summary,
        file,
        indent=2,
    )


# =====================================================================
# Final output
# =====================================================================
print()
print("=" * 70)
print("RIDGE EXTENSION RESULT")
print("=" * 70)

print(
    json.dumps(
        extended_selected_row,
        indent=2,
    )
)

print()
print(
    f"Saved recommendation: "
    f"{recommendation_path}"
)

if extended_selected_row["alpha"] == max(
    EXTENSION_ALPHAS
):
    print()
    print(
        "WARNING: the selected alpha remains on the upper "
        "search boundary. Extend the alpha values again. "
        "The pooled summaries are cached, so reservoir "
        "generation will not need to be repeated."
    )

else:
    print()
    print(
        "The selected alpha is inside the extended search "
        "range. No further Ridge extension is required."
    )


Candidate 1/3
units=1000, sr=0.99, lr=0.05, input scaling=0.2
Original selected alpha=1.0e+02, macro F1=60.1712%
Generating training summaries...
Generating validation summaries...
alpha=   1.0e+02 | accuracy= 62.65% | macro F1= 60.17% | balanced= 63.25%
alpha=   3.0e+02 | accuracy= 61.71% | macro F1= 59.67% | balanced= 63.56%
alpha=   1.0e+03 | accuracy= 59.70% | macro F1= 57.78% | balanced= 62.87%
alpha=   3.0e+03 | accuracy= 57.03% | macro F1= 55.27% | balanced= 61.57%
alpha=   1.0e+04 | accuracy= 54.09% | macro F1= 52.40% | balanced= 59.35%
Original versus repeated alpha=1.0e+02 macro F1 difference: 0.000000000000
Selected for this candidate: alpha=1.0e+02, macro F1=60.17%

Candidate 2/3
units=700, sr=0.8, lr=0.1, input scaling=0.1
Original selected alpha=1.0e+01, macro F1=57.5318%
Generating training summaries...
Generating validation summaries...
alpha=   1.0e+01 | accuracy= 61.07% | macro F1= 57.53% | balanced= 60.41%
alpha=   1.0e+02 | accuracy= 60.81% | macro F1= 57.48% | bal

In [13]:
# =====================================================================
# UPDATE LOCKED MANIFEST AFTER TARGETED RIDGE EXTENSION
# =====================================================================

from pathlib import Path
import json
import shutil


manifest_path = (
    RESULTS_DIR / "selected_reservoir_config.json"
)

recommendation_path = (
    RESULTS_DIR / "ridge_extension_recommendation.json"
)

backup_path = (
    RESULTS_DIR
    / "selected_reservoir_config_before_ridge_extension.json"
)


if not manifest_path.exists():
    raise FileNotFoundError(
        f"Missing selection manifest: {manifest_path}"
    )

if not recommendation_path.exists():
    raise FileNotFoundError(
        f"Missing Ridge extension recommendation: "
        f"{recommendation_path}"
    )


with open(manifest_path) as file:
    manifest = json.load(file)

with open(recommendation_path) as file:
    recommendation = json.load(file)


selected = recommendation["selected_candidate"]


# ---------------------------------------------------------------------
# Verify that the Ridge extension selected the same reservoir
# ---------------------------------------------------------------------
selected_reservoir = {
    "units": int(selected["units"]),
    "sr": float(selected["sr"]),
    "lr": float(selected["lr"]),
    "input_scaling": float(
        selected["input_scaling"]
    ),
    "rc_connectivity": float(
        manifest["reservoir_cfg"]["rc_connectivity"]
    ),
}


if selected_reservoir != manifest["reservoir_cfg"]:
    raise RuntimeError(
        "The Ridge extension selected a different reservoir. "
        "Do not update the manifest automatically.\n"
        f"Original: {manifest['reservoir_cfg']}\n"
        f"Extended: {selected_reservoir}"
    )


# ---------------------------------------------------------------------
# Create a backup before modifying the locked record
# ---------------------------------------------------------------------
if not backup_path.exists():
    shutil.copy2(
        manifest_path,
        backup_path,
    )

    print(
        f"Created manifest backup: {backup_path}"
    )
else:
    print(
        f"Manifest backup already exists: {backup_path}"
    )


# ---------------------------------------------------------------------
# Expand the recorded Ridge grid
# ---------------------------------------------------------------------
original_alphas = [
    float(alpha)
    for alpha in manifest[
        "search_design"
    ]["ridge_alphas"]
]

extension_alphas = [
    float(alpha)
    for alpha in recommendation[
        "extension_alphas"
    ]
]

complete_alpha_grid = sorted(
    set(
        original_alphas
        + extension_alphas
    )
)


manifest[
    "search_design"
]["ridge_alphas"] = complete_alpha_grid


# ---------------------------------------------------------------------
# Update the selected validation result
# ---------------------------------------------------------------------
manifest["selected_validation"] = {
    "acc": float(
        selected["val_acc"]
    ),
    "macro_f1": float(
        selected["val_macro_f1"]
    ),
    "balanced_acc": float(
        selected["val_balanced_acc"]
    ),
    "ridge_alpha": float(
        selected["alpha"]
    ),
    "stage": "stage3_ridge_extension",
}


# ---------------------------------------------------------------------
# Preserve full provenance for the targeted extension
# ---------------------------------------------------------------------
manifest["ridge_extension"] = {
    "completed": True,
    "selection_metric": recommendation[
        "selection_metric"
    ],
    "tested_extension_alphas": (
        extension_alphas
    ),
    "selected_alpha": float(
        selected["alpha"]
    ),
    "selected_alpha_is_boundary": (
        float(selected["alpha"])
        in (
            min(complete_alpha_grid),
            max(complete_alpha_grid),
        )
    ),
    "recommendation_path": str(
        recommendation_path.relative_to(PROJECT_ROOT)
    ),
    "alpha_curve_path": str(
        (RESULTS_DIR / "reservoir_stage3_ridge_extension_alpha_curves.csv").relative_to(PROJECT_ROOT)
    ),
    "candidate_results_path": str(
        (RESULTS_DIR / "reservoir_stage3_ridge_extension.csv").relative_to(PROJECT_ROOT)
    ),
}


# ---------------------------------------------------------------------
# Validate before saving
# ---------------------------------------------------------------------
if manifest[
    "ridge_extension"
]["selected_alpha_is_boundary"]:
    raise RuntimeError(
        "The selected alpha is still on the complete "
        "Ridge search boundary."
    )

if manifest["selected_validation"]["ridge_alpha"] != 100.0:
    raise RuntimeError(
        "Unexpected selected Ridge alpha."
    )

if manifest["selected_validation"]["stage"] != (
    "stage3_ridge_extension"
):
    raise RuntimeError(
        "Manifest stage was not updated correctly."
    )


# ---------------------------------------------------------------------
# Save the updated locked manifest
# ---------------------------------------------------------------------
with open(manifest_path, "w") as file:
    json.dump(
        manifest,
        file,
        indent=2,
    )


print()
print("=" * 70)
print("UPDATED LOCKED MANIFEST")
print("=" * 70)

print(
    json.dumps(
        {
            "reservoir_cfg": (
                manifest["reservoir_cfg"]
            ),
            "selected_validation": (
                manifest["selected_validation"]
            ),
            "ridge_alphas": (
                manifest[
                    "search_design"
                ]["ridge_alphas"]
            ),
            "ridge_extension": (
                manifest["ridge_extension"]
            ),
        },
        indent=2,
    )
)

print()
print(
    f"Updated manifest: {manifest_path}"
)

print(
    "The locked manifest now includes the completed "
    "Ridge extension."
)

Created manifest backup: /home/olliechandler/ESN-NSYNTH/results_sweep_final/selected_reservoir_config_before_ridge_extension.json

UPDATED LOCKED MANIFEST
{
  "reservoir_cfg": {
    "units": 1000,
    "sr": 0.99,
    "lr": 0.05,
    "input_scaling": 0.2,
    "rc_connectivity": 0.1
  },
  "selected_validation": {
    "acc": 0.6265183782931062,
    "macro_f1": 0.6017123878077728,
    "balanced_acc": 0.6324534914584826,
    "ridge_alpha": 100.0,
    "stage": "stage3_ridge_extension"
  },
  "ridge_alphas": [
    1e-10,
    1e-09,
    1e-08,
    1e-07,
    1e-06,
    1e-05,
    0.0001,
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    300.0,
    1000.0,
    3000.0,
    10000.0
  ],
  "ridge_extension": {
    "completed": true,
    "selection_metric": "macro_f1",
    "tested_extension_alphas": [
      100.0,
      300.0,
      1000.0,
      3000.0,
      10000.0
    ],
    "selected_alpha": 100.0,
    "selected_alpha_is_boundary": false,
    "recommendation_path": "results_swee